In [2]:
import numpy as np
from utils import *


In [3]:
import pandas as pd
df=pd.read_csv('spam_detection_dataset.csv')   

In [4]:
df.head()

,num_links,num_words,has_offer,sender_score,all_caps,is_spam
0,3,98,1,0.718607,0,0
1,0,170,0,0.698901,1,0
2,0,38,0,0.620466,0,0
3,0,116,0,0.701755,0,0
4,3,89,1,0.583621,1,1


In [7]:
df.shape


(20000, 6)

In [5]:
df.isnull().sum()   

num_links       0
num_words       0
has_offer       0
sender_score    0
all_caps        0
is_spam         0
dtype: int64

In [6]:
df.duplicated().sum()

402

In [8]:
# Drop duplicates
df.drop_duplicates(inplace=True)
# Check for duplicates again
df.duplicated().sum()

0

In [9]:
df.shape

(19598, 6)

In [10]:
df.columns

Index(['num_links', 'num_words', 'has_offer', 'sender_score', 'all_caps',
       'is_spam'],
      dtype='object')

In [11]:
df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 19598 entries, 0 to 19999
Data columns (total 6 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   num_links     19598 non-null  int64  
 1   num_words     19598 non-null  int64  
 2   has_offer     19598 non-null  int64  
 3   sender_score  19598 non-null  float64
 4   all_caps      19598 non-null  int64  
 5   is_spam       19598 non-null  int64  
dtypes: float64(1), int64(5)
memory usage: 1.0 MB


In [12]:
df.describe()

,num_links,num_words,has_offer,sender_score,all_caps,is_spam
count,19598.000000,19598.000000,19598.000000,19598.000000,19598.000000,19598.000000
mean,1.503980,109.555771,0.305286,0.687977,0.099347,0.093530
std,1.225113,52.021015,0.460540,0.185019,0.299135,0.291181
min,0.000000,20.000000,0.000000,0.000000,0.000000,0.000000
25%,1.000000,64.000000,0.000000,0.563561,0.000000,0.000000
50%,1.000000,110.000000,0.000000,0.694875,0.000000,0.000000
75%,2.000000,155.000000,1.000000,0.823769,0.000000,0.000000
max,9.000000,199.000000,1.000000,1.000000,1.000000,1.000000


About Dataset
This is a synthetic dataset for training and testing spam detection models. It contains 20,000 email samples, and each sample is described by five features and one label.

Features:
num_links

Type: Integer
Meaning: Number of links present in the email body
Generated using a Poisson distribution with an average (λ) of 1.5
Assumption: More links often mean higher chances of spam
num_words

Type: Integer
Meaning: Total number of words in the email
Randomly picked between 20 and 200
Assumption: Short or overly long emails might look suspicious, but this is more of a neutral feature
has_offer

Type: Binary (0 or 1)
Meaning: Whether the email contains the word “offer”
Simulated using a binomial distribution (30% chance of being 1)
Assumption: Marketing language like “offer” is common in spam
sender_score

Type: Float between 0 and 1
Meaning: A simulated reputation score of the email sender
Normally distributed around 0.7, clipped to stay between 0 and 1
Assumption: A low sender score means the sender is less trustworthy (and more likely to send spam)
all_caps

Type: Binary (0 or 1)
Meaning: Whether the subject line is written in ALL CAPS
Simulated with a 10% chance of being 1
Assumption: All-caps subject lines are usually attention-grabbing and common in spam
Target:
is_spam
Type: Binary (0 or 1)
Meaning: Whether the email is spam
Generated using a rule-based formula:
Spam probability increases if:
Links > 2
It contains an “offer”
Sender score < 0.4
Subject is in all caps
These factors are combined with different weights
A little noise is added using Gaussian randomness to simulate real-world uncertainty
Emails are labeled as spam if the final probability crosses 0.5


In [13]:
df['num_links'].value_counts()

num_links
1    6550
2    4962
0    4341
3    2471
4     900
5     278
6      80
7      11
8       4
9       1
Name: count, dtype: int64

In [14]:
df['num_words'].value_counts()

num_words
21     150
128    140
179    132
114    129
23     129
      ... 
50      91
136     90
58      89
85      89
53      85
Name: count, Length: 180, dtype: int64

In [15]:
df['has_offer'].value_counts()

has_offer
0    13615
1     5983
Name: count, dtype: int64

In [16]:
df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 19598 entries, 0 to 19999
Data columns (total 6 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   num_links     19598 non-null  int64  
 1   num_words     19598 non-null  int64  
 2   has_offer     19598 non-null  int64  
 3   sender_score  19598 non-null  float64
 4   all_caps      19598 non-null  int64  
 5   is_spam       19598 non-null  int64  
dtypes: float64(1), int64(5)
memory usage: 1.0 MB


In [17]:
df.describe()

,num_links,num_words,has_offer,sender_score,all_caps,is_spam
count,19598.000000,19598.000000,19598.000000,19598.000000,19598.000000,19598.000000
mean,1.503980,109.555771,0.305286,0.687977,0.099347,0.093530
std,1.225113,52.021015,0.460540,0.185019,0.299135,0.291181
min,0.000000,20.000000,0.000000,0.000000,0.000000,0.000000
25%,1.000000,64.000000,0.000000,0.563561,0.000000,0.000000
50%,1.000000,110.000000,0.000000,0.694875,0.000000,0.000000
75%,2.000000,155.000000,1.000000,0.823769,0.000000,0.000000
max,9.000000,199.000000,1.000000,1.000000,1.000000,1.000000


In [18]:
df['is_spam'].value_counts()

is_spam
0    17765
1     1833
Name: count, dtype: int64

In [19]:
!pip install imbalanced-learn

In [42]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
import matplotlib.pyplot as plt

# --- Logistic Regression Class ---
class LogisticRegression:
    def __init__(self, learning_rate=0.01, n_iters=1000, reg_lambda=0.1):
        self.lr = learning_rate
        self.n_iters = n_iters
        self.reg_lambda = reg_lambda
        self.weights = None
        self.bias = None
        self.losses = []
        self.val_losses = []
        self.train_accuracies = []
        self.val_accuracies = []

    def _sigmoid(self, x):
        return 1 / (1 + np.exp(-x))

    def compute_loss(self, y_true, y_pred):
        epsilon = 1e-9
        y1 = y_true * np.log(y_pred + epsilon)
        y2 = (1 - y_true) * np.log(1 - y_pred + epsilon)
        cross_entropy = -np.mean(y1 + y2)
        reg_term = (self.reg_lambda / 2) * np.sum(self.weights ** 2)
        return cross_entropy + reg_term

    def feed_forward(self, X):
        z = np.dot(X, self.weights) + self.bias
        return self._sigmoid(z)

    def fit(self, X_train, y_train):
        n_samples, n_features = X_train.shape
        self.weights = np.zeros(n_features)
        self.bias = 0

        for _ in range(self.n_iters):
            A = self.feed_forward(X_train)
            dz = A - y_train
            dw = (1 / n_samples) * np.dot(X_train.T, dz) + self.reg_lambda * self.weights
            db = (1 / n_samples) * np.sum(dz)

            self.weights -= self.lr * dw
            self.bias -= self.lr * db

            # # Store losses
            # train_loss = self.compute_loss(y_train, A)
            # self.losses.append(train_loss)

            # val_preds = self.feed_forward(X_val)
            # val_loss = self.compute_loss(y_val, val_preds)
            # self.val_losses.append(val_loss)

            # # Accuracies
            # train_pred = self.predict(X_train)
            # val_pred = self.predict(X_val)
            # self.train_accuracies.append(self.accuracy(y_train, train_pred))
            # self.val_accuracies.append(self.accuracy(y_val, val_pred))

    def predict(self, X):
        linear_output = np.dot(X, self.weights) + self.bias
        y_pred = self._sigmoid(linear_output)
        return np.array([1 if i > 0.5 else 0 for i in y_pred])

    def accuracy(self,y, y_hat):
        accuracy = np.sum(y == y_hat) / len(y)
        return accuracy









In [38]:
X = df.drop("is_spam", axis=1).values
y = df["is_spam"].values

In [39]:
type(X), type(y)

(numpy.ndarray, numpy.ndarray)

In [40]:
X_train, X_val, y_train, y_val = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)

In [ ]:
regressor = LogisticRegression(learning_rate=0.0001, n_iters=1000)
regressor.fit(X_train, y_train)
predictions = regressor.predict(X_val)
cm ,accuracy,sens,precision,f_score  = confusion_matrix(np.asarray(y_val), np.asarray(predictions))
print("Validation accuracy: {0:.3f}".format(accuracy))
print("Confusion Matrix:",np.array(cm))



Test accuracy: 0.906
Confusion Matrix: [[3553    0]
 [ 367    0]]


In [48]:
# Evaluation metrics from scratch
def compute_metrics(y_true, y_pred):
    TP = np.sum((y_true == 1) & (y_pred == 1))
    TN = np.sum((y_true == 0) & (y_pred == 0))
    FP = np.sum((y_true == 0) & (y_pred == 1))
    FN = np.sum((y_true == 1) & (y_pred == 0))

    accuracy = (TP + TN) / len(y_true)
    precision = TP / (TP + FP) if (TP + FP) else 0
    recall = TP / (TP + FN) if (TP + FN) else 0
    f1_score = 2 * (precision * recall) / (precision + recall) if (precision + recall) else 0

    print("\n--- Evaluation Metrics (from scratch) ---")
    print(f"Accuracy : {accuracy:.4f}")
    print(f"Precision: {precision:.4f}")
    print(f"Recall   : {recall:.4f}")
    print(f"F1 Score : {f1_score:.4f}")
    print("Confusion Matrix:")
    print(f"TP: {TP}, FP: {FP}, FN: {FN}, TN: {TN}")

# Call it
compute_metrics(y_val, predictions)



--- Evaluation Metrics (from scratch) ---
Accuracy : 0.9064
Precision: 0.0000
Recall   : 0.0000
F1 Score : 0.0000
Confusion Matrix:
TP: 0, FP: 0, FN: 367, TN: 3553


precision, recall, and F1-score are 0 because your model did not predict any positive (spam) class in the validation set — meaning all predictions were 0 (not spam).

➤ Why is this happening?
Because your dataset is imbalanced, and you are not using resampling (oversampling or undersampling) here. Your model prefers predicting only the majority class (not spam) because it leads to higher overall accuracy — but it completely fails to detect minority class (spam).

This is why:

Accuracy is misleading in imbalanced datasets.

You should use recall, precision, and F1-score to evaluate.



In [49]:
# Use lower threshold for classification
def predict_with_threshold(regressor, X, threshold=0.3):
    probs = regressor._sigmoid(np.dot(X, regressor.weights) + regressor.bias)
    return np.array([1 if i > threshold else 0 for i in probs])

y_val_pred = predict_with_threshold(regressor, X_val, threshold=0.3)
compute_metrics(y_val, y_val_pred)



--- Evaluation Metrics (from scratch) ---
Accuracy : 0.7939
Precision: 0.0984
Recall   : 0.1471
F1 Score : 0.1179
Confusion Matrix:
TP: 54, FP: 495, FN: 313, TN: 3058


Metric Explanations
Accuracy = 0.7939
→ About 79.4% of total predictions are correct.

Precision = 0.0984
→ Only 9.8% of the emails your model flagged as spam were actually spam.
(Low precision = high false positives)

Recall = 0.1471
→ Your model caught only 14.7% of actual spam messages.
(Low recall = many missed spam emails)

F1 Score = 0.1179
→ Harmonic mean of precision and recall. Low due to both precision and recall being low.

In [50]:
df.shape

(19598, 6)

# Using sklearn

In [51]:
df.head()

,num_links,num_words,has_offer,sender_score,all_caps,is_spam
0,3,98,1,0.718607,0,0
1,0,170,0,0.698901,1,0
2,0,38,0,0.620466,0,0
3,0,116,0,0.701755,0,0
4,3,89,1,0.583621,1,1


In [52]:
from sklearn.linear_model import LogisticRegression as SklearnLogisticRegression
from sklearn.metrics import classification_report, confusion_matrix

# Use the same training and validation set
sk_model = SklearnLogisticRegression(max_iter=1000)
sk_model.fit(X_train, y_train)

# Predict on validation set
y_val_pred_sk = sk_model.predict(X_val)

# Classification report
print("--- Evaluation Metrics (Scikit-learn) ---")
print(classification_report(y_val, y_val_pred_sk, target_names=["Not Spam", "Spam"]))

# Confusion Matrix
tn, fp, fn, tp = confusion_matrix(y_val, y_val_pred_sk).ravel()
print(f"Confusion Matrix:\nTP: {tp}, FP: {fp}, FN: {fn}, TN: {tn}")


--- Evaluation Metrics (Scikit-learn) ---
              precision    recall  f1-score   support

    Not Spam       0.96      0.98      0.97      3553
        Spam       0.76      0.56      0.65       367

    accuracy                           0.94      3920
   macro avg       0.86      0.77      0.81      3920
weighted avg       0.94      0.94      0.94      3920

Confusion Matrix:
TP: 207, FP: 64, FN: 160, TN: 3489


# 📊 Evaluation Metrics (Scikit-learn Logistic Regression)

### ✅ **Classification Report**

| Class      | Precision | Recall | F1-Score | Support |
|------------|-----------|--------|----------|---------|
| Not Spam   | 0.96      | 0.98   | 0.97     | 3553    |
| Spam       | 0.76      | 0.56   | 0.65     | 367     |

- **Accuracy**: 0.94  
- **Macro Average**:  
  - Precision: 0.86  
  - Recall: 0.77  
  - F1-score: 0.81  
- **Weighted Average**:  
  - Precision: 0.94  
  - Recall: 0.94  
  - F1-score: 0.94

---

### 🧮 **Confusion Matrix**

|               | Predicted: Not Spam | Predicted: Spam |
|---------------|---------------------|-----------------|
| Actual: Not Spam | TN = 3489           | FP = 64         |
| Actual: Spam     | FN = 160            | TP = 207        |

---

### 🧠 **Metric Definitions**
- **Precision (Spam)** = TP / (TP + FP) = 207 / (207 + 64) ≈ **0.76**  
- **Recall (Spam)** = TP / (TP + FN) = 207 / (207 + 160) ≈ **0.56**  
- **F1-Score (Spam)** = 2 × (Precision × Recall) / (Precision + Recall) ≈ **0.65**  
- **Accuracy** = (TP + TN) / (TP + TN + FP + FN) = (207 + 3489) / 3920 ≈ **0.94**

---

### 📌 **Interpretation**
- The model performs **very well overall** with high accuracy.
- It achieves **strong precision for spam** (few false positives), though **recall is moderate** (some spam is missed).
- **F1-score balances both precision and recall**, and 0.65 indicates good spam detection ability.
- The model is highly effective at recognizing "Not Spam" (majority class), with near-perfect metrics.

---

### 📈 **Suggestions**
- If your goal is **not to miss spam messages** (i.e. higher recall), consider:
  - Adjusting the classification threshold
  - Using class weights (`class_weight='balanced'`)
  - Trying different models (e.g., Random Forest, XGBoost)



Validation Accuracy: 0.5121024486349564


# ❓ Why is Accuracy Not Always a Suitable Evaluation Metric?

## ⚠️ Problem with Accuracy in Imbalanced Datasets

Accuracy = (TP + TN) / (Total Samples)

While accuracy tells you how many predictions were correct, **it can be misleading**, especially when the dataset is **imbalanced** — where one class significantly outnumbers the other.

---

## 📉 Example Scenario

Imagine a dataset where:
- 95% of emails are **not spam**
- 5% of emails are **spam**

A model that **always predicts "Not Spam"** would be:
- 95% accurate
- But it would **never detect spam** (Recall = 0)

---

## 🧪 When Accuracy Fails

- **Imbalanced Data**: High accuracy even if the model misses all instances of the minority class.
- **Misclassification Cost**: In domains like healthcare or fraud detection, false negatives can be far more costly than false positives.
- **Class Importance**: Sometimes detecting the minority class (e.g., spam, fraud, disease) is the actual goal.

---

## ✅ Better Alternatives

- **Precision**: How many predicted positives are truly positive?
- **Recall (Sensitivity)**: How many actual positives were correctly predicted?
- **F1 Score**: Harmonic mean of precision and recall.
- **ROC-AUC**: Measures the tradeoff between TPR and FPR.

---

## 📝 Summary

> Accuracy can be deceptive in imbalanced or high-stakes scenarios.  
> Always complement it with precision, recall, F1-score, or ROC-AUC depending on the domain.



# 🎯 When to Choose Precision Over Recall (and Vice-Versa)

Understanding whether to prioritize **precision** or **recall** depends on the specific problem and its consequences.

---

## ✅ Precision

**Definition**:  
> Precision = TP / (TP + FP)  
> Of all predicted positives, how many are actually positive?

**Choose Precision When**:
- **False positives are costly**.
- You want **high confidence in your positive predictions**.
- You can **tolerate missing some positives**, but you don’t want to wrongly label negatives as positives.

### 📌 Examples:
- **Spam Filter**: Better to let a few spam emails through than mark important ones as spam.
- **Cancer Diagnosis (Initial Test)**: You don’t want to scare patients with a false positive.

---

## ✅ Recall

**Definition**:  
> Recall = TP / (TP + FN)  
> Of all actual positives, how many did we correctly identify?

**Choose Recall When**:
- **False negatives are costly or dangerous**.
- You want to **catch as many positives as possible**, even if some predictions are wrong.
- You can **tolerate some false positives**.

### 📌 Examples:
- **Disease Screening**: Missing a positive case could be life-threatening.
- **Security Systems**: Better to raise more false alarms than miss a threat.

---

## ⚖️ F1 Score: The Balance

- Use **F1 Score** when you want a balance between precision and recall.
- Especially useful in **imbalanced datasets**.

---

## 📝 Summary Table

| Situation                        | Metric to Focus On |
|----------------------------------|--------------------|
| False positives are worse        | Precision          |
| False negatives are worse        | Recall             |
| Need balance                     | F1 Score           |



SyntaxError: invalid character '’' (U+2019) (763425093.py, line 16)